In [ ]:
import sys
sys.path.append("../../")

In [2]:
# imports
import pandas as pd
from tensorflow.keras import layers, Model
# utilities
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

from utils.processing import filter_rsos_by_min_state_count, rso_train_test_split, assemble_scaled_state_df

In [3]:
# Separate features into different categories
X_numerical = ['INCL', 'RAAN', 'ECC', 'ARG_PER', 'MEAN_MOTION', 'SMA_KM','APOGEE_KM', 'PERIGEE_KM', 'MEAN_MOTION_1ST_DER'] 
X_numerical_revised = X_numerical + ['MEAN_MOTION_1ST_DER', 'B_STAR']
X_features_to_use = X_numerical
response = 'TYPE'

df_final = pd.read_csv('../data/large/final_df_v1.csv') # <-- Run Part 1 of DataConsolidation.ipynb to produce file
df_final = df_final.sort_values(by=['NUMBER', 'EPOCH'])
df_final.reset_index(inplace=True, drop=True)

### Step 1: Reduce Objects to Minimum Number of States

In [4]:
STATE_COUNT_THRESHOLD = 20 # <-- should tune this
rsos_meet_threshold, _ = filter_rsos_by_min_state_count(df_final, STATE_COUNT_THRESHOLD)

### Step 2: test-train split on objects then states

In [5]:
df_objects_train, df_objects_test = rso_train_test_split(df_final, rsos_meet_threshold)
df_states_scaled_train, scaler = assemble_scaled_state_df(df_final, df_objects_train['NUMBER'],X_features_to_use, STATE_COUNT_THRESHOLD, StandardScaler(), fit_scaler=True)
df_states_scaled_test, _ = assemble_scaled_state_df(df_final, df_objects_test['NUMBER'],X_features_to_use, STATE_COUNT_THRESHOLD, scaler)

#print(df_states_scaled_train.shape[0]/df_objects_train.shape[0]) # STATE_COUNT_THRESHOLD
#print(df_states_scaled_test.shape[0]/df_objects_test.shape[0])

### Step 3: 

### reshape into a usable form


In [19]:
N_OBJECTS = df_objects_train.shape[0]
N_FEAT = len(X_features_to_use)

X_train_states = np.zeros((N_OBJECTS, STATE_COUNT_THRESHOLD, N_FEAT))
X_train_debug_numbers =[]
y_train = []
for obj_idx in range(N_OBJECTS):
    obj_number = df_objects_train['NUMBER'][obj_idx]
    tmp = df_states_scaled_train[df_states_scaled_train['NUMBER']==obj_number].reset_index(drop=True)
    X_train_states[obj_idx] = tmp[X_features_to_use]
    X_train_debug_numbers.append(obj_number)
    y_train.append(tmp['TYPE'][0])

In [21]:
y_train_encoded = LabelEncoder().fit_transform(y_train)
len(y_train_encoded)

18528

In [18]:
X_train_states.shape

(18528, 20, 9)

In [22]:
#X_train = np.random.rand(10000, STATE_COUNT_THRESHOLD, N_FEAT).astype(np.float32)  # or float64 if needed
#y_train = np.random.randint(0, 3, size=(10000,))

def single_state_evaluation_model(): 
    '''given features of ONE state, returns softmax probs'''

    inputs = layers.Input(shape=(N_FEAT,))
    x = layers.Dense(64, activation='relu')(inputs)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(3, activation='softmax')(x)
    return Model(inputs, outputs, name="StateModel")


def create_object_model(state_model):
    '''per object model'''
    # STATE_COUNT_THRESHOLD states per object, each of shape (N_FEAT,)
    inputs = layers.Input(shape=(STATE_COUNT_THRESHOLD, N_FEAT))
    # apply the state model to each of the STATE_COUNT_THRESHOLD states
    state_probs = layers.TimeDistributed(state_model)(inputs)  # Output shape: (20, 3)

    # classify the object based on these STATE_COUNT_THRESHOLD softmaxed vectors
    x = layers.Flatten()(state_probs)  # Shape: (3*STATE_COUNT_THRESHOLD,)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(3, activation='softmax')(x)

    return Model(inputs, outputs, name="ObjectClassifier")


state_model = single_state_evaluation_model()
object_model = create_object_model(state_model)

object_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# `X` is (num_objects, STATE_COUNT_THRESHOLD, N_FEAT) and `y` is (num_objects,)
object_model.fit(X_train_states, y_train_encoded, batch_size=32, epochs=3)




2025-05-06 14:12:58.967242: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Max
2025-05-06 14:12:58.967290: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 48.00 GB
2025-05-06 14:12:58.967295: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 18.00 GB
I0000 00:00:1746555178.967345 107823023 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1746555178.967404 107823023 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Epoch 1/3


2025-05-06 14:12:59.608892: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


579/579 ━━━━━━━━━━━━━━━━━━━━ 14s 21ms/step - accuracy: 0.6956 - loss: 0.7011
Epoch 2/3
579/579 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.5737 - loss: 0.9752
Epoch 3/3
579/579 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.6028 - loss: 1.3619
